# PV labels and negative net load

This notebook links GIGI customer labels to meter points, builds a per-customer asset-additions timeline, and scans 15-minute measurements for negative net load. It prints coverage and category counts at each stage. The full 77 GiB monthly scan is deliberately opt-in after the OBIS register semantics have been verified.

In [1]:
from pathlib import Path
from collections import defaultdict
import json
import re

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 30)
pd.set_option('display.max_colwidth', 120)

def find_data_root():
    starts = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    for start in starts:
        candidate = start / 'store' / 'input_data'
        if candidate.exists():
            return candidate
    raise FileNotFoundError('Could not find store/input_data. Start the notebook from the repository or a child directory.')

DATA_ROOT = find_data_root()
GIGI_PATH = DATA_ROOT / 'HackDays2026 - GIGI.csv'
ZGP_PATH = DATA_ROOT / 'Zähler-GP.csv'
MAPPING_PATH = DATA_ROOT / 'mpid_zähler_mapping.csv'

print(f'Data root: {DATA_ROOT}')
for path in (GIGI_PATH, ZGP_PATH, MAPPING_PATH):
    print(f'  {path.name}: {path.exists()}')

Data root: /home/renku/work/store/input_data
  HackDays2026 - GIGI.csv: True
  Zähler-GP.csv: True
  mpid_zähler_mapping.csv: True


In [2]:
# Load and inspect the three small reference tables. utf-8-sig removes their BOM.
gigi_raw = pd.read_csv(GIGI_PATH, sep=';', encoding='utf-8-sig', dtype='string')
zgp_raw = pd.read_csv(ZGP_PATH, sep=';', encoding='utf-8-sig', dtype='string')
mapping_raw = pd.read_csv(MAPPING_PATH, sep=';', encoding='utf-8-sig', dtype='string')

def clean_text(series):
    return series.astype('string').str.strip()

def normalise_name(name):
    return re.sub(r'\s+', ' ', str(name).strip()).casefold()

for name, frame in {'GIGI': gigi_raw, 'Zähler-GP': zgp_raw, 'MP mapping': mapping_raw}.items():
    print(f'\n{name}: {len(frame):,} rows × {len(frame.columns)} columns')
    print('Columns:', list(frame.columns))

print('\nDuplicate-key checks:')
print('  GIGI nonblank GP-Nr duplicates:', gigi_raw['GP-Nr'].dropna().astype(str).str.strip().duplicated().sum())
print('  Mapping duplicate MP IDs:', mapping_raw['MP ID'].dropna().astype(str).str.strip().duplicated().sum())
print('  Zähler-GP duplicate designations:', zgp_raw['Zählpunktbezeichnung'].dropna().astype(str).str.strip().duplicated().sum())


GIGI: 1,192 rows × 14 columns
Columns: ['GP-Nr', 'PLZ', 'Ort', 'Kanton', 'WärmePumpe', ' PV', 'PV-Leistung in kWp ', 'Batterie/Speicher', 'Ladestation für Elektrofahrzeuge', 'Wärmepumpenboiler', 'Datum Unterschrift', 'geplanter Baustart', 'Übergabe', 'InBetrieb-Datum']

Zähler-GP: 89,910 rows × 3 columns
Columns: ['Zählpunktbezeichnung', 'GPartner', 'Anlage']

MP mapping: 89,993 rows × 2 columns
Columns: ['MP ID', 'Zählpunktbezeichnung']

Duplicate-key checks:
  GIGI nonblank GP-Nr duplicates: 289
  Mapping duplicate MP IDs: 0
  Zähler-GP duplicate designations: 136


In [3]:
# Consolidate source GIGI rows to the requested customer grain and create dated asset additions.
gigi = gigi_raw.copy()
gigi['_source_row'] = np.arange(len(gigi), dtype=int)
gigi['gp_nr'] = clean_text(gigi['GP-Nr'])
gigi = gigi[gigi['gp_nr'].notna() & gigi['gp_nr'].ne('')].copy()

column_lookup = {normalise_name(c): c for c in gigi.columns}
def source_column(label):
    return column_lookup.get(normalise_name(label))

asset_fields = {
    'heat_pump': source_column('WärmePumpe'),
    'pv': source_column('PV'),
    'battery_storage': source_column('Batterie/Speicher'),
    'ev_charger': source_column('Ladestation für Elektrofahrzeuge'),
    'heat_pump_boiler': source_column('Wärmepumpenboiler'),
}
asset_fields = {asset: col for asset, col in asset_fields.items() if col is not None}
if 'pv' not in asset_fields:
    raise KeyError(f'Could not find the PV field. Available columns: {list(gigi.columns)}')

def is_x(value):
    return pd.notna(value) and str(value).strip().casefold() == 'x'

gigi['pv_positive_row'] = gigi[asset_fields['pv']].map(is_x)
inbetrieb_col = source_column('InBetrieb-Datum')
uebergabe_col = source_column('Übergabe')
gigi['inbetrieb_date'] = pd.to_datetime(gigi[inbetrieb_col], errors='coerce', dayfirst=True) if inbetrieb_col else pd.NaT
gigi['uebergabe_date'] = pd.to_datetime(gigi[uebergabe_col], errors='coerce', dayfirst=True) if uebergabe_col else pd.NaT
gigi['event_date'] = gigi['inbetrieb_date'].fillna(gigi['uebergabe_date'])
gigi['date_source'] = np.where(gigi['inbetrieb_date'].notna(), 'InBetrieb-Datum',
                               np.where(gigi['uebergabe_date'].notna(), 'Übergabe', 'unknown'))

def customer_timeline(customer_rows):
    ordered = customer_rows.sort_values(['event_date', '_source_row'], na_position='last')
    seen, events = set(), {}
    for _, row in ordered.iterrows():
        present = {asset for asset, col in asset_fields.items() if is_x(row[col])}
        added = sorted(present - seen)
        seen |= present
        if not added:
            continue
        date = None if pd.isna(row['event_date']) else row['event_date'].date().isoformat()
        key = (date, row['date_source'])
        event = events.setdefault(key, {'date': date, 'assets_added': [], 'date_source': row['date_source']})
        event['assets_added'].extend(asset for asset in added if asset not in event['assets_added'])
    return list(events.values())

timeline = (gigi.groupby('gp_nr', sort=False)
              .apply(customer_timeline, include_groups=False)
              .rename('asset_additions').reset_index())
customer_base = (gigi.groupby('gp_nr', as_index=False)
                 .agg(gigi_record_count=('_source_row', 'size'),
                      pv_positive=('pv_positive_row', 'any'),
                      inbetrieb_dates=('inbetrieb_date', lambda s: sorted({d.date().isoformat() for d in s.dropna()})),
                      uebergabe_dates=('uebergabe_date', lambda s: sorted({d.date().isoformat() for d in s.dropna()}))))
customer_base = customer_base.merge(timeline, on='gp_nr', how='left')
customer_base['asset_additions'] = customer_base['asset_additions'].map(json.dumps)
pv_customers = customer_base[customer_base['pv_positive']].copy()

print(f'Usable GIGI customer IDs: {len(customer_base):,}')
print(f'PV-positive customers (any normalized x): {len(pv_customers):,}')
print('PV raw values:')
display(gigi[asset_fields['pv']].fillna('<blank>').value_counts(dropna=False).rename_axis('PV raw value').to_frame('rows'))
display(pv_customers[['gp_nr', 'gigi_record_count', 'inbetrieb_dates', 'uebergabe_dates', 'asset_additions']].head(10))

Usable GIGI customer IDs: 878
PV-positive customers (any normalized x): 724
PV raw values:


,rows
PV raw value,
x,729
-,306
<blank>,108
X,24


,gp_nr,gigi_record_count,inbetrieb_dates,uebergabe_dates,asset_additions
1,104479,1,[2024-04-15],[2024-05-13],"[{""date"": ""2024-04-15"", ""assets_added"": [""battery_storage"", ""pv""], ""date_source"": ""InBetrieb-Datum""}]"
2,104987,1,[2023-11-03],[],"[{""date"": ""2023-11-03"", ""assets_added"": [""battery_storage"", ""pv""], ""date_source"": ""InBetrieb-Datum""}]"
3,106914,1,[2025-03-06],[2025-03-14],"[{""date"": ""2025-03-06"", ""assets_added"": [""battery_storage"", ""pv""], ""date_source"": ""InBetrieb-Datum""}]"
4,107868,2,"[2024-09-06, 2026-04-08]","[2024-10-01, 2026-04-17]","[{""date"": ""2024-09-06"", ""assets_added"": [""pv""], ""date_source"": ""InBetrieb-Datum""}, {""date"": ""2026-04-08"", ""assets_ad..."
5,108398,1,[2019-03-06],[],"[{""date"": ""2019-03-06"", ""assets_added"": [""pv""], ""date_source"": ""InBetrieb-Datum""}]"
6,108801,2,[2021-05-11],"[2021-06-08, 2022-04-27]","[{""date"": ""2021-05-11"", ""assets_added"": [""heat_pump"", ""pv"", ""battery_storage""], ""date_source"": ""InBetrieb-Datum""}]"
7,108808,1,[],[2024-07-15],"[{""date"": ""2024-07-15"", ""assets_added"": [""battery_storage"", ""pv""], ""date_source"": ""\u00dcbergabe""}]"
10,112352,2,[],[2022-02-18],"[{""date"": ""2022-02-18"", ""assets_added"": [""battery_storage"", ""pv""], ""date_source"": ""\u00dcbergabe""}, {""date"": null, ""..."
11,113592,1,[2024-06-28],[2024-07-10],"[{""date"": ""2024-06-28"", ""assets_added"": [""battery_storage"", ""ev_charger"", ""pv""], ""date_source"": ""InBetrieb-Datum""}]"
13,116373,1,[],[2021-06-30],"[{""date"": ""2021-06-30"", ""assets_added"": [""battery_storage"", ""pv""], ""date_source"": ""\u00dcbergabe""}]"


In [4]:
# Link each PV-positive customer to every mapped meter point and quantify ambiguity.
zgp = zgp_raw.rename(columns={'GPartner': 'gp_nr', 'Zählpunktbezeichnung': 'designation', 'Anlage': 'anlage'}).copy()
zgp['gp_nr'] = clean_text(zgp['gp_nr'])
zgp['designation'] = clean_text(zgp['designation'])
zgp['anlage'] = clean_text(zgp['anlage'])
mapping = mapping_raw.rename(columns={'MP ID': 'mp_id', 'Zählpunktbezeichnung': 'designation'}).copy()
mapping['mp_id'] = clean_text(mapping['mp_id'])
mapping['designation'] = clean_text(mapping['designation'])

designation_rows = zgp.groupby('designation', dropna=False).size().rename('designation_row_count')
links = (pv_customers[['gp_nr']].merge(zgp[['gp_nr', 'designation', 'anlage']], on='gp_nr', how='left')
         .merge(mapping[['designation', 'mp_id']], on='designation', how='left'))
links['designation_row_count'] = links['designation'].map(designation_rows).fillna(0).astype(int)
links = links[links['mp_id'].notna() & links['mp_id'].ne('')].copy()
links = links.drop_duplicates(['gp_nr', 'mp_id', 'designation', 'anlage'])
mp_customer_count = links.groupby('mp_id')['gp_nr'].nunique().rename('mp_customer_count')
links['mp_customer_count'] = links['mp_id'].map(mp_customer_count)
links['mapping_ambiguous'] = (links['mp_customer_count'].gt(1) | links['designation_row_count'].gt(1))

link_summary = (links.groupby('gp_nr', as_index=False)
                .agg(meter_point_count=('mp_id', 'nunique'),
                     mapping_ambiguity_count=('mapping_ambiguous', 'sum'),
                     mapped_anlagen=('anlage', lambda s: sorted(set(s.dropna())))))
customer_table = pv_customers.merge(link_summary, on='gp_nr', how='left')
customer_table['meter_point_count'] = customer_table['meter_point_count'].fillna(0).astype(int)
customer_table['mapping_ambiguity_count'] = customer_table['mapping_ambiguity_count'].fillna(0).astype(int)
customer_table['mapping_ambiguity'] = customer_table['mapping_ambiguity_count'].gt(0)
customer_table['mapped_anlagen'] = customer_table['mapped_anlagen'].map(lambda x: x if isinstance(x, list) else [])

print(f'PV customers with at least one mapped meter point: {(customer_table.meter_point_count > 0).sum():,}')
print(f'PV customers without a mapped meter point: {(customer_table.meter_point_count == 0).sum():,}')
print(f'PV customers with mapping ambiguity: {customer_table.mapping_ambiguity.sum():,}')
display(customer_table[['gp_nr', 'meter_point_count', 'mapping_ambiguity', 'mapped_anlagen', 'asset_additions']].head(10))

PV customers with at least one mapped meter point: 284
PV customers without a mapped meter point: 440
PV customers with mapping ambiguity: 0


,gp_nr,meter_point_count,mapping_ambiguity,mapped_anlagen,asset_additions
0,104479,2,False,"[463340, 539681]","[{""date"": ""2024-04-15"", ""assets_added"": [""battery_storage"", ""pv""], ""date_source"": ""InBetrieb-Datum""}]"
1,104987,1,False,[538467],"[{""date"": ""2023-11-03"", ""assets_added"": [""battery_storage"", ""pv""], ""date_source"": ""InBetrieb-Datum""}]"
2,106914,1,False,[547625],"[{""date"": ""2025-03-06"", ""assets_added"": [""battery_storage"", ""pv""], ""date_source"": ""InBetrieb-Datum""}]"
3,107868,2,False,"[464220, 465201]","[{""date"": ""2024-09-06"", ""assets_added"": [""pv""], ""date_source"": ""InBetrieb-Datum""}, {""date"": ""2026-04-08"", ""assets_ad..."
4,108398,1,False,[519504],"[{""date"": ""2019-03-06"", ""assets_added"": [""pv""], ""date_source"": ""InBetrieb-Datum""}]"
5,108801,1,False,[520292],"[{""date"": ""2021-05-11"", ""assets_added"": [""heat_pump"", ""pv"", ""battery_storage""], ""date_source"": ""InBetrieb-Datum""}]"
6,108808,1,False,[539577],"[{""date"": ""2024-07-15"", ""assets_added"": [""battery_storage"", ""pv""], ""date_source"": ""\u00dcbergabe""}]"
7,112352,1,False,[522821],"[{""date"": ""2022-02-18"", ""assets_added"": [""battery_storage"", ""pv""], ""date_source"": ""\u00dcbergabe""}, {""date"": null, ""..."
8,113592,1,False,[539594],"[{""date"": ""2024-06-28"", ""assets_added"": [""battery_storage"", ""ev_charger"", ""pv""], ""date_source"": ""InBetrieb-Datum""}]"
9,116373,0,False,[],"[{""date"": ""2021-06-30"", ""assets_added"": [""battery_storage"", ""pv""], ""date_source"": ""\u00dcbergabe""}]"


In [5]:
# Discover monthly files and profile OBIS codes from a bounded sample before selecting registers.
monthly_files = sorted(DATA_ROOT.rglob('LG_AIM2Hackerdays_kWh_*.csv'))
print(f'Discovered monthly files: {len(monthly_files)} (expected: 43)')
if monthly_files:
    print('First file:', monthly_files[0])
    print('Last file: ', monthly_files[-1])
if len(monthly_files) != 43:
    print('WARNING: file count differs from the documented 43 monthly exports.')

PROFILE_FILES = 1
PROFILE_ROWS_PER_FILE = 200_000
obis_profile = []
for path in monthly_files[:PROFILE_FILES]:
    sample = pd.read_csv(path, sep=';', usecols=['OBIS-Code'], nrows=PROFILE_ROWS_PER_FILE, dtype='string')
    obis_profile.append(sample['OBIS-Code'].value_counts(dropna=False).rename(path.name))
if obis_profile:
    display(pd.concat(obis_profile, axis=1).fillna(0).astype(int))

print('\nDo not assume OBIS semantics from this profile. Configure verified register codes in the next cell.')

Discovered monthly files: 43 (expected: 43)
First file: /home/renku/work/store/input_data/2023/2023/April 2023/LG_AIM2Hackerdays_kWh_20260826_201441.csv
Last file:  /home/renku/work/store/input_data/2026/März 2026/LG_AIM2Hackerdays_kWh_20260728_063802.csv


ValueError: Usecols do not match columns, columns expected but not found: ['OBIS-Code']

In [ ]:
# Set these only after validating OBIS semantics and units.
# For separate registers, list every code representing interval import/export.
NET_LOAD_MODE = 'separate'  # 'separate' or 'signed'
IMPORT_OBIS_CODES = set()
EXPORT_OBIS_CODES = set()
SIGNED_NET_OBIS_CODES = set()

# Safety switch: scanning all exports is ~77 GiB. Set True after configuration.
RUN_MONTHLY_SCAN = False
CHUNK_ROWS = 100_000

if NET_LOAD_MODE == 'separate' and not (IMPORT_OBIS_CODES and EXPORT_OBIS_CODES):
    print('Scan not configured: supply verified import and export OBIS code sets.')
elif NET_LOAD_MODE == 'signed' and not SIGNED_NET_OBIS_CODES:
    print('Scan not configured: supply verified signed-net OBIS code set.')
else:
    print(f'Configured {NET_LOAD_MODE} net-load scan.')

In [ ]:
# Stream monthly files. This cell is intentionally inert until the preceding configuration is complete.
def to_long_intervals(frame, time_columns):
    long = frame.melt(id_vars=['MP ID', 'Datum', 'OBIS-Code'], value_vars=time_columns,
                      var_name='interval', value_name='value')
    long['value'] = pd.to_numeric(long['value'].astype('string').str.replace(',', '.', regex=False), errors='coerce')
    return long

def monthly_net_evidence(path, mp_ids):
    header = pd.read_csv(path, sep=';', nrows=0)
    time_columns = [c for c in header.columns if re.fullmatch(r'\d{2}:\d{2}', str(c))]
    usecols = ['MP ID', 'OBIS-Code', 'Datum', *time_columns]
    codes = (IMPORT_OBIS_CODES | EXPORT_OBIS_CODES) if NET_LOAD_MODE == 'separate' else SIGNED_NET_OBIS_CODES
    kept = []
    for chunk in pd.read_csv(path, sep=';', usecols=usecols, chunksize=CHUNK_ROWS, dtype='string'):
        chunk = chunk[chunk['MP ID'].astype('string').str.strip().isin(mp_ids)]
        chunk = chunk[chunk['OBIS-Code'].isin(codes)]
        if not chunk.empty:
            kept.append(chunk)
    if not kept:
        return pd.DataFrame(columns=['mp_id', 'valid_interval_count', 'negative_interval_count', 'first_negative_timestamp', 'first_negative_value', 'first_negative_source'])
    long = to_long_intervals(pd.concat(kept, ignore_index=True), time_columns)
    key = ['MP ID', 'Datum', 'interval']
    if NET_LOAD_MODE == 'separate':
        imported = long[long['OBIS-Code'].isin(IMPORT_OBIS_CODES)].groupby(key, as_index=False)['value'].sum(min_count=1).rename(columns={'value': 'import'})
        exported = long[long['OBIS-Code'].isin(EXPORT_OBIS_CODES)].groupby(key, as_index=False)['value'].sum(min_count=1).rename(columns={'value': 'export'})
        net = imported.merge(exported, on=key, how='outer')
        net['net_load'] = net['import'] - net['export']
        net = net[net['import'].notna() & net['export'].notna()].copy()
    else:
        net = long.groupby(key, as_index=False)['value'].sum(min_count=1).rename(columns={'value': 'net_load'}).dropna(subset=['net_load'])
    net['timestamp_label'] = net['Datum'].astype(str) + ' ' + net['interval'].astype(str)
    net['is_negative'] = net['net_load'].lt(0)
    result = net.groupby('MP ID', as_index=False).agg(valid_interval_count=('net_load', 'size'), negative_interval_count=('is_negative', 'sum'))
    first = (net[net['is_negative']].sort_values(['MP ID', 'Datum', 'interval']).groupby('MP ID', as_index=False).first()[['MP ID', 'timestamp_label', 'net_load']])
    result = result.merge(first, on='MP ID', how='left').rename(columns={'MP ID': 'mp_id', 'timestamp_label': 'first_negative_timestamp', 'net_load': 'first_negative_value'})
    result['first_negative_source'] = path.name
    return result

configured = ((NET_LOAD_MODE == 'separate' and IMPORT_OBIS_CODES and EXPORT_OBIS_CODES) or
              (NET_LOAD_MODE == 'signed' and SIGNED_NET_OBIS_CODES))
scan_summary = pd.DataFrame()
if not RUN_MONTHLY_SCAN:
    print('Monthly scan skipped. Set RUN_MONTHLY_SCAN = True only after configuring verified OBIS codes.')
elif not configured:
    raise ValueError('Configure verified OBIS codes before enabling the monthly scan.')
else:
    pv_mp_ids = set(links['mp_id'].dropna().astype(str))
    summaries = []
    for number, path in enumerate(monthly_files, start=1):
        summary = monthly_net_evidence(path, pv_mp_ids)
        summaries.append(summary)
        print(f'[{number}/{len(monthly_files)}] {path.name}: {len(summary):,} meter points with valid intervals')
    scan_summary = pd.concat(summaries, ignore_index=True) if summaries else pd.DataFrame()
    print(f'Finished scan. Monthly meter-point summaries: {len(scan_summary):,}')

In [ ]:
# Roll monthly meter-point evidence to the requested one-row-per-customer result.
if scan_summary.empty:
    print('No scan evidence yet. Configure registers and run the preceding cell to populate final categories.')
else:
    mp_evidence = (scan_summary.groupby('mp_id', as_index=False)
                   .agg(valid_interval_count=('valid_interval_count', 'sum'),
                        negative_interval_count=('negative_interval_count', 'sum'),
                        first_negative_timestamp=('first_negative_timestamp', 'min'),
                        first_negative_value=('first_negative_value', 'min')))
    customer_evidence = (links[['gp_nr', 'mp_id']].drop_duplicates()
                         .merge(mp_evidence, on='mp_id', how='left')
                         .groupby('gp_nr', as_index=False)
                         .agg(valid_interval_count=('valid_interval_count', 'sum'),
                              negative_interval_count=('negative_interval_count', 'sum'),
                              first_negative_timestamp=('first_negative_timestamp', 'min'),
                              first_negative_value=('first_negative_value', 'min')))
    customer_table = customer_table.merge(customer_evidence, on='gp_nr', how='left')
    customer_table['valid_interval_count'] = customer_table['valid_interval_count'].fillna(0).astype('int64')
    customer_table['negative_interval_count'] = customer_table['negative_interval_count'].fillna(0).astype('int64')
    customer_table['negative_load_status'] = np.select(
        [customer_table['meter_point_count'].eq(0), customer_table['valid_interval_count'].eq(0), customer_table['negative_interval_count'].gt(0)],
        ['unlinked', 'linked_but_unobserved', 'observed_with_negative'],
        default='observed_without_negative')
    print('Customer categories:')
    display(customer_table['negative_load_status'].value_counts().rename_axis('category').to_frame('customers'))
    print('Mapping ambiguity:')
    display(customer_table['mapping_ambiguity'].value_counts().rename_axis('mapping_ambiguous').to_frame('customers'))
    display(customer_table.sort_values(['negative_load_status', 'gp_nr']).head(20))

## Interpretation limits

A PV-labelled customer may not show negative net load when simultaneous consumption exceeds generation. Missing/unpaired intervals, ambiguous mappings, uncertain asset dates, and data ending in July 2026 also limit conclusions. This notebook does not filter measurements by asset dates yet; it retains the asset-additions timeline for that later analysis.